# TA-03 — Preprocessing Data Kartu Kredit
## Tahap 2: Persiapan data untuk Logistic Regression

Notebook ini melanjutkan EDA. Letakkan `UCI_Credit_Card.csv` di folder yang sama dengan notebook.

**Target:** `default.payment.next.month`  
- `0` = tidak gagal bayar
- `1` = gagal bayar

Catatan: kolom `ID` dihapus dari fitur, bukan harus dihapus dari file CSV asli.

## 1. Import library dan baca dataset

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from pathlib import Path

FILE_DATASET = Path("UCI_Credit_Card.csv")
if not FILE_DATASET.exists():
    raise FileNotFoundError(
        "UCI_Credit_Card.csv tidak ditemukan. Simpan CSV di folder yang sama dengan notebook."
    )

df = pd.read_csv(FILE_DATASET)

print("Ukuran dataset:", df.shape)
display(df.head())

Ukuran dataset: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


## 2. Memeriksa target dan kolom

In [2]:
target = "default.payment.next.month"

print("Kolom target tersedia:", target in df.columns)
print("Jumlah nilai target yang hilang:", df[target].isna().sum())
print("Nilai unik target:", sorted(df[target].dropna().unique()))

print("\nJumlah nilai kosong per kolom:")
display(df.isnull().sum().to_frame("Jumlah nilai kosong"))

Kolom target tersedia: True
Jumlah nilai target yang hilang: 0
Nilai unik target: [np.int64(0), np.int64(1)]

Jumlah nilai kosong per kolom:


,Jumlah nilai kosong
ID,0
LIMIT_BAL,0
SEX,0
EDUCATION,0
MARRIAGE,0
AGE,0
PAY_0,0
PAY_2,0
PAY_3,0
PAY_4,0


## 3. Menyiapkan fitur dan target

In [3]:
# Hapus baris yang targetnya kosong jika ada.
df_model = df.dropna(subset=[target]).copy()

# Pisahkan target dari fitur dan keluarkan ID dari fitur prediksi.
X = df_model.drop(columns=["ID", target], errors="ignore")
y = df_model[target].astype(int)

print("Jumlah baris untuk pemodelan:", len(df_model))
print("Jumlah fitur sebelum preprocessing:", X.shape[1])
print("Distribusi target:")
display(y.value_counts().sort_index().rename(index={0: "Tidak gagal bayar", 1: "Gagal bayar"}).to_frame("Jumlah"))

Jumlah baris untuk pemodelan: 30000
Jumlah fitur sebelum preprocessing: 23
Distribusi target:


,Jumlah
default.payment.next.month,
Tidak gagal bayar,23364
Gagal bayar,6636


## 4. Menentukan fitur numerik dan kategorikal

Kolom `SEX`, `EDUCATION`, dan `MARRIAGE` adalah fitur kategorikal. Kolom `PAY_0`, `PAY_2`–`PAY_6` adalah kode status pembayaran yang juga diperlakukan sebagai kategori agar model tidak mengasumsikan jarak numerik yang sama antarkode. Fitur tagihan, pembayaran, batas kredit, dan usia diperlakukan sebagai numerik.

In [4]:
categorical_features = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"
]

categorical_features = [col for col in categorical_features if col in X.columns]
numeric_features = [col for col in X.columns if col not in categorical_features]

print("Fitur kategorikal:", categorical_features)
print(f"Jumlah fitur kategorikal: {len(categorical_features)}")
print(f"Jumlah fitur numerik: {len(numeric_features)}")
print("\nNilai unik pada fitur kategorikal:")
for col in categorical_features:
    print(f"{col}: {sorted(X[col].dropna().unique().tolist())}")

Fitur kategorikal: ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
Jumlah fitur kategorikal: 9
Jumlah fitur numerik: 14

Nilai unik pada fitur kategorikal:
SEX: [1, 2]
EDUCATION: [0, 1, 2, 3, 4, 5, 6]
MARRIAGE: [0, 1, 2, 3]
PAY_0: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_2: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_3: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_4: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_5: [-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]
PAY_6: [-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]


## 5. Membagi data train dan test (80:20)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nProporsi target pada data latih:")
display((y_train.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("Persentase (%)"))

print("Proporsi target pada data uji:")
display((y_test.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("Persentase (%)"))

X_train: (24000, 23)
X_test : (6000, 23)
y_train: (24000,)
y_test : (6000,)

Proporsi target pada data latih:


,Persentase (%)
default.payment.next.month,
0,77.88
1,22.12


Proporsi target pada data uji:


,Persentase (%)
default.payment.next.month,
0,77.88
1,22.12


## 6. Membuat pipeline preprocessing

- Fitur numerik: imputasi median jika diperlukan, lalu StandardScaler.
- Fitur kategorikal: imputasi modus jika diperlukan, lalu OneHotEncoder.
- `handle_unknown='ignore'` membantu aplikasi menerima kategori yang tidak ditemukan saat training tanpa langsung error.

Pipeline akan di-fit hanya pada data latih untuk menghindari data leakage.

In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Bentuk data latih setelah preprocessing:", X_train_processed.shape)
print("Bentuk data uji setelah preprocessing:", X_test_processed.shape)
print("Preprocessing berhasil.")

Bentuk data latih setelah preprocessing: (24000, 91)
Bentuk data uji setelah preprocessing: (6000, 91)
Preprocessing berhasil.


## 7. Memeriksa kesiapan data

Pada tahap ini, data sudah dipisahkan menjadi train/test dan preprocessing sudah dipelajari dari data latih. Belum ada model yang dilatih pada notebook ini.

In [7]:
print("Ringkasan:")
print(f"- Total data: {len(X):,}")
print(f"- Data latih: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"- Data uji: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"- Jumlah fitur asli setelah ID dikeluarkan: {X.shape[1]}")
print(f"- Jumlah fitur setelah encoding: {X_train_processed.shape[1]}")
print(f"- Missing values pada target: {y.isna().sum()}")

Ringkasan:
- Total data: 30,000
- Data latih: 24,000 (80.0%)
- Data uji: 6,000 (20.0%)
- Jumlah fitur asli setelah ID dikeluarkan: 23
- Jumlah fitur setelah encoding: 91
- Missing values pada target: 0


## Kesimpulan preprocessing

Setelah menjalankan seluruh sel, catat ukuran data latih dan data uji serta jumlah fitur setelah encoding untuk laporan. Berikutnya kita akan membandingkan model Logistic Regression dasar dengan model yang menangani ketidakseimbangan kelas. Jika menggunakan SMOTE, SMOTE hanya diterapkan pada data latih, bukan data uji.